In [ ]:
#!/usr/bin/env python3
"""Convert per-painting confusion matrices from .npz to the long CSV that
`analysis_compositional.R` reads.

The validation script writes `per_painting_confusion.npz` (a stack of 15x15
integer matrices, one per held-out artwork). R cannot read .npz, and the
painting-level bootstrap of the correction matrix needs one row per non-zero
cell. This script is the bridge between the two halves of the pipeline.

Usage
    python convert_confusion.py                       # uses env vars / defaults
    python convert_confusion.py --in path/to.npz --out path/to.csv

Environment
    PAINTINGS_OUT   directory holding per_painting_confusion.npz (default: outputs)
"""
from __future__ import annotations

import argparse
import os
import sys

import numpy as np
import pandas as pd


def convert(npz_path: str, csv_path: str) -> pd.DataFrame:
    with np.load(npz_path, allow_pickle=False) as z:
        for key in ("stems", "mats", "names"):
            if key not in z:
                raise KeyError(f"{npz_path} is missing array '{key}'")
        stems = [str(s) for s in z["stems"]]
        mats = z["mats"]
        names = [str(n) for n in z["names"]]

    n_paintings, n_true, n_pred = mats.shape
    if n_true != n_pred:
        raise ValueError(f"confusion matrices are not square: {mats.shape}")
    if n_true != len(names):
        raise ValueError(f"{n_true} classes in matrices but {len(names)} names")
    if n_paintings != len(stems):
        raise ValueError(f"{n_paintings} matrices but {len(stems)} stems")

    records = []
    for stem, matrix in zip(stems, mats):
        rows, cols = np.nonzero(matrix)
        for i, j in zip(rows, cols):
            records.append((stem, names[i], names[j], int(matrix[i, j])))

    long_df = pd.DataFrame(records,
                           columns=["stem", "true_class", "pred_class", "n"])

    # Integrity: the long form must preserve every pixel.
    if long_df["n"].sum() != int(mats.sum()):
        raise AssertionError("pixel total changed during conversion")
    if long_df["stem"].nunique() != n_paintings:
        raise AssertionError("an artwork was lost during conversion")

    long_df.to_csv(csv_path, index=False)
    print(f"wrote {csv_path}: {len(long_df)} rows, {n_paintings} artworks, "
          f"{long_df['n'].sum():,} pixels")
    return long_df


def main(argv=None) -> int:
    out_dir = os.environ.get("PAINTINGS_OUT", "outputs")
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--in", dest="npz",
                        default=os.path.join(out_dir, "per_painting_confusion.npz"))
    parser.add_argument("--out", dest="csv",
                        default=os.path.join(out_dir, "per_painting_confusion_long.csv"))
    args = parser.parse_args(argv)

    if not os.path.exists(args.npz):
        parser.error(f"input not found: {args.npz}")
    convert(args.npz, args.csv)
    return 0


if __name__ == "__main__":
    sys.exit(main())